In [1]:
import matplotlib
%matplotlib qt

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy

#
from utils import ComputeROIs

Autosaving every 180 seconds


In [2]:
#
fname = r"D:\User Training\Readtest8105\Image_001_001.raw"

# 
gr = ComputeROIs(fname)

#
std_map = gr.make_std_map()

memmap :  (60000, 512, 512)
data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0


In [3]:
#
rois, indexes = gr.find_roi_boundaries(std_map)

In [4]:
#
gr.show_contour_map(std_map, indexes)

In [38]:
#
gr.scale=4000
gr.subsample = 10
roi_traces = gr.compute_traces()

memmap :  (60000, 512, 512)


100%|███████████████████████████████████████████████████████████████████████████████████████| 56/56 [00:07<00:00,  7.79it/s]


In [67]:
#
ensemble1 = [20,27]
ensemble2 = [28,43]

#
E1 = roi_traces[ensemble1[0]]+roi_traces[ensemble1[1]]
E2 = roi_traces[ensemble2[0]]+roi_traces[ensemble2[1]]
diff = E1-E2
print ("diff: ", diff.shape)
y = np.histogram(diff, bins=np.arange(np.min(diff),np.max(diff),100))


def find_reward_thresholds(diff, gr):
    
    error = 0.1
    low = -1050
    high = 1750
    n_rewards = 0
    
    # use 30 sec windows to see max # of random rewards
    increment = int(30/gr.subsample)
    print (" a second in frames: ", increment)
    n_sec_recording = int(diff.shape[0]/increment)
    print ("nsec recording: ", n_sec_recording)
    # use 30 sec lockouts
    n_rewards_random = n_sec_recording //30
    print ("# random rewards possible (every 30sec) ", n_rewards_random)
    
    # set the low and high to match this
    while n_rewards<n_rewards_random:
        
        #
        k=0
        n_rewards = 0
        while k<diff.shape[0]:
            if diff[k]>high or diff[k]<low:
                # reward state reached
                
                # implement a lockout of 5 seconds
                n_rewards+=1
                k+= int(5*increment)
            else:
                k+=1
            
        #if n_rewards > n_rewards_random:
        #    break
    
        if n_rewards<n_rewards_random:
            low*=0.98
            high*=0.98
        
        print ("updated rwards #: ", n_rewards, low, high)

    return low, high

# find 30% reward threshold
low, high = find_reward_thresholds(diff, gr)
print ("thresholds: ", low, high)
#
plt.figure()
ax=plt.subplot(121)
plt.plot(y[1][:-1],y[0]/np.max(y[0]))
ax2 = ax.twiny()
ax2.plot(cumsum,c='red')
plt.plot([thresh_low, thresh_low], [0,1], '--', c='grey')
plt.plot([thresh_high, thresh_high], [0,1], '--', c='grey')

#
ax=plt.subplot(122)
t = np.arange(diff.shape[0])/gr.subsample
plt.plot([t[0],t[-1]], [low, low], '--', c='grey')
plt.plot([t[0],t[-1]], [high, high], '--', c='grey')
plt.plot(t,E1,c='blue',alpha=.1)
plt.plot(t,E2,c='red',alpha=.1)
plt.plot(t, diff,c='black')
plt.show()



diff:  (6000,)
 a second in frames:  3
nsec recording:  2000
# random rewards possible (every 30sec)  66
updated rwards #:  64 -1029.0 1715.0
updated rwards #:  65 -1008.42 1680.7
updated rwards #:  66 -1008.42 1680.7
thresholds:  -1008.42 1680.7


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)

